# 01 — Pips, points, ticks and pip value

[Module 02](../course/02-trading-terminology.md) · [Notebook index](README.md)

**Purpose:** Measure price movement and its monetary value. All prices, conversion rates and contract sizes in this notebook are hypothetical. Run the cells in order.

- Pip: conventional FX unit, usually 0.0001 for non-JPY pairs and 0.01 for JPY-quoted pairs.
- Point: MT5 display unit (often 0.00001 on five-decimal EURUSD).
- Tick size: minimum permitted price increment for an instrument; it can differ from point size. A tick also means a quote update, depending on context.
- A standard Forex lot commonly represents 100,000 base-currency units; verify broker specifications.

For linear Forex contracts:

$$\text{Movement in units}=\frac{\text{End price}-\text{Start price}}{\text{Unit size}}$$

$$\text{Pip value in account currency}=\text{Lots}\times\text{Contract size}\times\text{Pip size}\times\text{Conversion rate}$$

The conversion rate is account-currency units per one quote-currency unit. Use 1 when currencies match. This simplified conversion uses a supplied rate, not live prices. Tick value for other linear instruments can be supplied directly from broker specifications rather than inferred from FX pip conventions.


In [ ]:
from math import isfinite, isclose

def positive(value, name):
    if not isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be finite and positive")
    return value

def finite(value, name):
    if not isfinite(value):
        raise ValueError(f"{name} must be finite")
    return value

def expect_value_error(function, *args):
    try:
        function(*args)
    except ValueError:
        return
    raise AssertionError("Expected ValueError")

def price_movement(start, end, unit_size):
    positive(start, "start")
    positive(end, "end")
    positive(unit_size, "unit size")
    return (end - start) / unit_size

def forex_pip_value(lots, contract_size, pip_size, quote_to_account=1.0):
    return (positive(lots, "lots") * positive(contract_size, "contract size")
            * positive(pip_size, "pip size") * positive(quote_to_account, "conversion rate"))

def tick_value_for_position(lots, tick_value_per_lot):
    return positive(lots, "lots") * positive(tick_value_per_lot, "tick value per lot")


## Worked example

EUR/USD moves from 1.10000 to 1.10250. That is 25 pips, or 250 points when the point size is 0.00001. For 0.10 lots and a 100,000-unit contract, one pip is $1; at an illustrative 18 ZAR per USD, it is R18. At unchanged EUR/USD bid 1.10000 and ask 1.10020, the spread is 2 pips.


In [ ]:
print(f"Movement: {price_movement(1.10000, 1.10250, 0.0001):.1f} pips")
print(f"Movement: {price_movement(1.10000, 1.10250, 0.00001):.0f} points")
print(f"Spread: {price_movement(1.10000, 1.10020, 0.0001):.1f} pips")
print(f"Pip value (USD): {forex_pip_value(0.10, 100_000, 0.0001):.2f}")
print(f"Pip value (ZAR): {forex_pip_value(0.10, 100_000, 0.0001, 18):.2f}")


## Try it yourself

For USD/JPY, use 0.01 as the pip size: how many pips is a move from 150.00 to 149.80? For 0.10 lots, a 100,000-unit contract and an illustrative conversion rate of 1/150 USD per JPY, calculate the USD value of a pip. Expected values: −20 pips and approximately $0.67.


In [ ]:
assert isclose(price_movement(1.10000, 1.10250, 0.0001), 25)
assert isclose(price_movement(1.10000, 1.10250, 0.00001), 250)
assert isclose(price_movement(150.00, 149.80, 0.01), -20)
assert isclose(forex_pip_value(0.10, 100_000, 0.01, 1/150), 2/3)
assert isclose(forex_pip_value(0.10, 100_000, 0.0001, 18), 18)
assert isclose(tick_value_for_position(0.2, 5), 1)
expect_value_error(price_movement, 1.1, 1.2, 0)
expect_value_error(forex_pip_value, -1, 100_000, 0.0001)
expect_value_error(forex_pip_value, 1, 100_000, 0.0001, float("nan"))
print("All pip and tick checks passed.")


## Online calculator and limitations

- [Myfxbook pip calculator](https://www.myfxbook.com/forex-calculators/pip-calculator)
- [MT5 Market Watch and symbol specifications](https://www.metatrader5.com/en/terminal/help/trading/market_watch)
- [MQL5 profit estimation](https://www.mql5.com/en/book/automation/experts/experts_ordercalcprofit)

Compare like-for-like inputs: online tools may use live rates. Do not assume point = tick size, or a Forex pip convention applies to gold, indices or synthetic markets. Tick values may vary with price, conversion or profit/loss direction; use the correct broker value. These educational functions do not validate broker volume steps or calculate live order risk.
